In [ ]:
from urllib.request import urlopen
from deepagents import create_deep_agent
from langchain_quickjs import CodeInterpreterMiddleware
from deepagents.backends import StateBackend
from langgraph.checkpoint.memory import InMemorySaver


In [ ]:
from pathlib import Path
from typing import Dict, Any
from deepagents.backends.utils import create_file_data

def get_skills_from_path(path: str) -> Dict[str, Any]:
    base_path = Path(path)
    skills = {}
    
    if base_path.exists() and base_path.is_dir():
        for p in base_path.rglob("*"):
            if p.is_file() and p.name.lower() == "skill.md":
                # Extract relative folder names (e.g., 'aws', 'kubernetes')
                skill_folder = p.parent.relative_to(base_path).as_posix()
                
                virtual_path = f"/skills/{skill_folder}/SKILL.md"
                
                skills[virtual_path] = create_file_data(p.read_text("utf-8"))
                
    return skills

# Execution
skill_files = get_skills_from_path("skills")
skill_files

In [ ]:
d_agent = create_deep_agent(
    model="groq:Openai/gpt-oss-20b",
    backend = StateBackend(),
    skills=["/skills/"],
    checkpointer=InMemorySaver()
)
result = d_agent.invoke({
    "messages": [
        {
        "role": "user", 
        "content": "what skills do you have and when do you use each provide each skills you have"
        }
    ],
    "files": skill_files
},
config={"configurable": {"thread_id": "4"}}
)

In [5]:
result = d_agent.invoke({
    "messages": [
        {
        "role": "user", 
        "content": "how do i build a langgrapgh with conditional routing and memory, show a minimal example"
        }
    ],
    "files": skill_files
},
config={"configurable": {"thread_id": "4"}}
)

In [7]:
print(result)
for m in result["messages"]:
    m.pretty_print()

{'messages': [HumanMessage(content='what skills do you have and when do you use each provide each skills you have', additional_kwargs={}, response_metadata={}, id='01db28e0-8ea9-44dc-9183-26f39534aa5c'), AIMessage(content='**Available Skills**\n\n| Skill | What it does | When to use it |\n|-------|--------------|----------------|\n| **aws** | Design, deploy, and troubleshoot AWS solutions (core services, IaC, security, cost‑optimization). | When the user asks about AWS architecture, CloudFormation/Terraform, Lambda, ECS/EKS, IAM, cost‑analysis, or debugging AWS‑related issues. |\n| **kubernetes** | Deploy and manage containerized workloads on Kubernetes (workloads, networking, storage, config, security, observability). | When the user needs help writing or debugging `Deployment`, `Service`, `Ingress`, `ConfigMap`, `Secret`, or cluster‑level operations. |\n| **langgraph** | Build stateful agent workflows with LangGraph (graphs, nodes, edges, state, checkpointing, human‑in‑the‑loop). | W

In [ ]:
import textwrap
import re

def pretty_print_ai_response(response_input, width=90):
    response_text = ""

    # 1. Parse LangGraph / LangChain Trace input types securely
    if isinstance(response_input, dict):
        if "messages" in response_input and response_input["messages"]:
            for msg in reversed(response_input["messages"]):
                content = getattr(msg, 'content', '') or (msg.get('content', '') if isinstance(msg, dict) else '')
                if content.strip():
                    response_text = content
                    break
        
        if not response_text and "files" in response_input:
            for file_path, file_data in response_input["files"].items():
                if isinstance(file_data, dict) and "content" in file_data:
                    response_text = file_data["content"]
                    print(f"\n--- Displaying File: {file_path} ---")
                    break
                elif isinstance(file_data, str):
                    response_text = file_data
                    print(f"\n--- Displaying File: {file_path} ---")
                    break
    
    elif hasattr(response_input, 'content'):
        response_text = response_input.content
        
    elif isinstance(response_input, str):
        response_text = response_input
    
    if not response_text or not response_text.strip():
        print("⚠️ No printable text content found.")
        return

    # --- STRUCTURAL TEXT RENDER ENGINE ---
    lines = response_text.splitlines()
    processed_lines = []
    table_buffer = []
    in_table = False
    in_code_block = False

    def flush_table(buffer):
        if not buffer: return []
        cleaned_rows = []
        for row in buffer:
            cells = [cell.strip() for cell in row.split('|')]
            if row.strip().startswith('|'): cells = cells[1:]
            if row.strip().endswith('|'): cells = cells[:-1]
            cleaned_rows.append(cells)
            
        if not cleaned_rows: return []
        max_cols = max(len(row) for row in cleaned_rows)
        col_widths = [0] * max_cols
        
        for row in cleaned_rows:
            if all(set(cell).issubset({'-', ':', ' '}) for cell in row): continue
            for i, cell in enumerate(row):
                if i < len(col_widths):
                    col_widths[i] = max(col_widths[i], len(cell))
                    
        formatted_table = []
        for row in cleaned_rows:
            if all(set(cell).issubset({'-', ':', ' '}) for cell in row):
                divider_cells = ['-' * (col_widths[i] + 2) if i < len(col_widths) else '-----' for i in range(max_cols)]
                formatted_table.append("  |" + "|".join(divider_cells) + "|")
                continue
            padded_cells = [f" {row[i].ljust(col_widths[i])} " if i < len(row) else f" {''.ljust(col_widths[i])} " for i in range(max_cols)]
            formatted_table.append("  |" + "|".join(padded_cells) + "|")
        return formatted_table

    for line in lines:
        stripped_line = line.strip()

        # Handle code blocks
        if stripped_line.startswith("```"):
            if in_table:
                processed_lines.extend(flush_table(table_buffer))
                table_buffer, in_table = [], False
            in_code_block = not in_code_block
            processed_lines.append(line)
            continue

        if in_code_block:
            processed_lines.append(line)
            continue

        # Handle Markdown Tables safely (must have cell wrappers or divider notation)
        is_table_line = '|' in line and (stripped_line.startswith('|') or '---' in line or len(line.split('|')) > 2)
        if is_table_line:
            in_table = True
            table_buffer.append(line)
        else:
            if in_table:
                processed_lines.extend(flush_table(table_buffer))
                table_buffer, in_table = [], False
            
            if not stripped_line:
                processed_lines.append('')
            # Protect structural components (headers, list dots, dashes) from textwrap corruption
            elif re.match(r'^(#+|\s*[-*+•]\s|\s*\d+\.\s)', line):
                processed_lines.append(line)
            else:
                processed_lines.append(textwrap.fill(line, width=width))
                
    if in_table:
        processed_lines.extend(flush_table(table_buffer))

    print('\n'.join(processed_lines))
